# MOMENT fcst trên PTB-XL: LiRA + RMIA (TV4)

Notebook này chạy **sau** `moment_fcst_colab.ipynb` (forecasting lookback 512, horizon 96). Nó train K shadow model forecasting theo `dataset/splits/shadow_masks_<level>_K8_s42.npz` (cùng mask với bản cls), rồi tính **LiRA** (online, offline) và **RMIA** (offline, online) cho target model, ghi `results/foundation/fcst/<attack>_full.json`.

**Thiết kế cho forecasting** (khác bản cls vì không có xác suất lớp đúng):
- Thống kê của mỗi bản ghi là MSE trung bình trên 12 lead × 96 bước.
- **LiRA** dùng `φ = -log(MSE)`: MSE dương và lệch phải, còn log-MSE gần Gaussian, đúng giả định của LiRA.
- **RMIA** cần một "likelihood" nằm trong [0,1], nên dùng `p(x) = exp(-MSE(x)/τ)` (likelihood Gauss, bỏ hằng số chuẩn hoá). `τ` là **median MSE của các shadow OUT**, một hằng số toàn cục, chỉ dùng thông tin phía attacker. Có bảng độ nhạy theo τ ở cuối phần tấn công. Đây là điều chỉnh của nhóm cho bài toán regression, không có trong bài báo gốc.

**Điều kiện**
1. Đã chạy xong `moment_fcst_colab.ipynb` cùng `LEVEL`: có `ckpt/foundation_fcst/full_<level>_s42.done.json` và `full_<level>_s42_preds_ep10.pt`.
2. `EPOCHS`, `BATCH`, `MAX_LR`, `LOOKBACK`, `HORIZON` ở cell cấu hình phải **giống hệt** notebook fcst. Notebook tự kiểm tra và dừng nếu lệch.
3. Drive có `Final_Project/moment/`, `Final_Project/dataset/` (có `splits/` với file shadow masks và `archive (3).zip`).
4. GPU A100 hoặc L4. Mỗi shadow tốn xấp xỉ một lần train target, K=8 khoảng 8 lần. Session bị ngắt thì Run all lại, shadow xong được bỏ qua, shadow dở chạy tiếp từ checkpoint.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch touches CUDA
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
MOMENT_SRC = f'{DRIVE_ROOT}/moment'          # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'        # shared split script + splits/
PTBXL_ZIP = f'{DATASET_DIR}/archive (3).zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/foundation/fcst'

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy source to local disk (faster than installing from Drive), then install
!rm -rf /content/moment && cp -r {MOMENT_SRC} /content/moment
!pip install -q /content/moment wfdb scikit-learn pandas tqdm scipy

## 1. Cấu hình
Phần train phải khớp `moment_colab.ipynb`. `K` là số shadow dùng (tối đa 8, chẵn). Mỗi mẫu có K/2 shadow IN và K/2 shadow OUT.

In [ ]:
SEED = 42
LEVEL = 'record'            # record-level split, same as the target run
MODEL_NAME = 'AutonLab/MOMENT-1-large'
MODE = 'full'

# training recipe: MUST equal moment_fcst_colab.ipynb
EPOCHS = 10
BATCH = 16
MAX_LR = 1e-4
LOOKBACK, HORIZON = 512, 96
TARGET_EPOCH = 10           # epoch whose predictions the target attacks use (last of EVAL_AT)

K = 8                       # shadows to train (first K masks of the 8 in the npz)
CKPT_EVERY = 2              # resumable checkpoint of the shadow in progress (~4.5GB)
N_BOOT = 1000
TAU_MULTS = [0.5, 1.0, 2.0] # RMIA sensitivity; the reported attack uses 1.0 (fixed in advance)

CKPT_DIR = f'{DRIVE_ROOT}/ckpt/foundation_fcst'
PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
MASK_PATH = f'{DATASET_DIR}/splits/shadow_masks_{LEVEL}_K8_s{SEED}.npz'
CACHE_DIR = f'/content/cache/fcst_{LEVEL}_s{SEED}'

## 2. Dữ liệu và kiểm tra target

In [ ]:
import glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))

In [ ]:
import json, random, time
import numpy as np, torch, pandas as pd
from momentfm.data.ptbxl_classification_dataset import PTBXL_dataset

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
control_randomness()

assert os.path.exists(SPLIT_PATH) and os.path.exists(MASK_PATH), f'missing split/mask files in {DATASET_DIR}/splits'
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
assert 'shadow' in meta, 'meta has no shadow entry; run make_split.py with --shadow-k 8'
RMIA_A, RMIA_GAMMA = meta['shadow']['rmia_defaults']['a'], meta['shadow']['rmia_defaults']['gamma']

# the target must come from moment_fcst_colab.ipynb with exactly this recipe
TARGET_KEY = f"fcst|{LOOKBACK}|{HORIZON}|{MODEL_NAME}|{LEVEL}|{SEED}|{EPOCHS}|{BATCH}|{MAX_LR}|{meta['sha256'][:12]}"
tag_t = f'{MODE}_{LEVEL}_s{SEED}'
done_t = f'{CKPT_DIR}/{tag_t}.done.json'
assert os.path.exists(done_t), f'run moment_fcst_colab.ipynb (LEVEL={LEVEL}) first: missing {done_t}'
assert json.load(open(done_t))['key'] == TARGET_KEY, 'EPOCHS/BATCH/MAX_LR/split differ from the target run'
tgt = torch.load(f'{CKPT_DIR}/{tag_t}_preds_ep{TARGET_EPOCH}.pt')
print('target preds', tgt['pm'].shape, tgt['pn'].shape)

class Config:
    base_path = PTBXL_DIR
    cache_dir = CACHE_DIR
    path_to_pids = SPLIT_PATH
    load_cache = True
    fs = 100
    output_type = 'single'
    seq_len = LOOKBACK + HORIZON
os.makedirs(CACHE_DIR, exist_ok=True)
member_ds = PTBXL_dataset(Config, phase='train')
val_ds    = PTBXL_dataset(Config, phase='val')
nonmem_ds = PTBXL_dataset(Config, phase='test')
assert len(member_ds) == len(tgt['pm']) and len(nonmem_ds) == len(tgt['pn'])

class PoolDS(torch.utils.data.Dataset):
    def __init__(self, x): self.timeseries = x
    def __len__(self): return len(self.timeseries)
    def __getitem__(self, i): return self.timeseries[i], 0

# pool = every single-label record = train + val + test of the split; order: member, val, non-member
parts = [member_ds, val_ds, nonmem_ds]
pool_ids = np.concatenate([np.asarray(d.ecg_ids()) for d in parts])
pool_ds = PoolDS(np.concatenate([d.timeseries for d in parts]).astype(np.float32))
n_m, n_v, n_n, N = len(member_ds), len(val_ds), len(nonmem_ds), len(pool_ids)

mk = np.load(MASK_PATH)
order = np.searchsorted(mk['ids'], pool_ids)
assert len(mk['ids']) == N and (mk['ids'][order] == pool_ids).all(), 'mask ids differ from the dataset pool'
masks_pool = mk['masks'][:K][:, order]              # (K, N) True = shadow k trained on this record
ref_pool = np.flatnonzero(np.isin(pool_ids, mk['ref_ids']))
print('pool', N, 'K', K, 'in-fraction', masks_pool.mean().round(3), 'ref ids in pool', len(ref_pool))

## 3. Train K shadow model (forecasting)
Mỗi shadow train trên các bản ghi có mask = True (khoảng một nửa pool) bằng đúng công thức của target (MSE, lookback 512, horizon 96), rồi lưu **MSE từng bản ghi** trên toàn pool vào Drive (vài trăm KB mỗi shadow).

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from momentfm import MOMENTPipeline

dev = 'cuda'
bf16 = torch.cuda.get_device_capability()[0] >= 8
amp = lambda: torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=bf16)

def build_model(name):
    model = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs={
        'task_name': 'forecasting', 'forecast_horizon': HORIZON,
        'freeze_encoder': False, 'freeze_embedder': False, 'freeze_head': False,
        'enable_gradient_checkpointing': True})
    model.init()
    print(f'[{name}] trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
    return model.to(dev)

def split_xy(x):
    x = x.to(dev).float()
    return x[..., :LOOKBACK], x[..., LOOKBACK:]

def forward_fcst(model, x_in):
    mask = torch.ones(x_in.shape[0], x_in.shape[-1], device=dev)
    return model(x_enc=x_in, input_mask=mask).forecast.float()

@torch.no_grad()
def get_mse(model, ds, bs=64):
    model.eval()
    out = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False):
        x_in, y_tgt = split_xy(x)
        with amp():
            pred = forward_fcst(model, x_in)
        out.append(((pred - y_tgt) ** 2).mean(dim=(1, 2)).cpu())
    return torch.cat(out)                       # [N] per-record MSE

def save_atomic(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)   # a disconnect mid-write never corrupts the previous checkpoint
    # Force Drive FUSE to flush to server — prevents silent write loss on large files
    with open(path, 'rb') as _f:
        os.fsync(_f.fileno())
    assert os.path.getsize(path) > 256, f'save_atomic: {path} suspiciously small after write'

def train_shadow(j):
    tag = f'shadow_{LEVEL}_s{SEED}_k{j}'
    out_path, ckpt_path = f'{CKPT_DIR}/{tag}_mse.pt', f'{CKPT_DIR}/{tag}.pt'
    key = f"{TARGET_KEY}|shadow{j}|{meta['shadow']['sha256'][:12]}"
    if os.path.exists(out_path) and torch.load(out_path)['key'] == key:
        print(f'[shadow {j}] already finished, skipping'); return

    control_randomness(SEED + 1 + j)          # different head init / data order per shadow
    model = build_model(f'shadow {j}')
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    idx = np.flatnonzero(masks_pool[j])
    loader = DataLoader(Subset(pool_ds, idx), batch_size=BATCH, shuffle=True, drop_last=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=MAX_LR / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MAX_LR, total_steps=EPOCHS * len(loader))
    mse = torch.nn.MSELoss()
    start = 1

    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location='cpu')
        if ck['key'] == key:
            model.load_state_dict(ck['model'], strict=False)
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            start = ck['epoch'] + 1
            print(f"[shadow {j}] resumed after epoch {ck['epoch']}")
        else:
            print(f'[shadow {j}] checkpoint has a different config, ignoring it')
        del ck

    for ep in range(start, EPOCHS + 1):
        model.train()
        losses = []
        for x, _ in tqdm(loader, desc=f'shadow {j} ep{ep}/{EPOCHS}', leave=False):
            opt.zero_grad(set_to_none=True)
            x_in, y_tgt = split_xy(x)
            with amp():
                pred = forward_fcst(model, x_in)
            loss = mse(pred, y_tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step()
            losses.append(loss.item())
        print(f'[shadow {j}] ep{ep} train_loss {np.mean(losses):.4f}')
        if ep % CKPT_EVERY == 0 and ep < EPOCHS:
            save_atomic({'key': key, 'epoch': ep, 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                         'model': {k: v.cpu() for k, v in model.state_dict().items() if k in trainable}}, ckpt_path)

    ms = get_mse(model, pool_ds)
    m = masks_pool[j]
    print(f'[shadow {j}] mean MSE in {ms.numpy()[m].mean():.4f} out {ms.numpy()[~m].mean():.4f}')
    save_atomic({'key': key, 'mse': ms, 'mse_in': float(ms.numpy()[m].mean()), 'mse_out': float(ms.numpy()[~m].mean())}, out_path)
    if os.path.exists(ckpt_path): os.remove(ckpt_path)
    del model, opt; torch.cuda.empty_cache()

for j in range(K):
    t0 = time.time()
    train_shadow(j)
    print(f'shadow {j} done in {(time.time() - t0) / 60:.1f} min')

## 4. LiRA và RMIA cho forecasting
Score luôn theo quy ước **cao hơn = nhiều khả năng là member**.

- **LiRA**: `φ = -log(MSE)` của bản ghi. Với mỗi bản ghi, fit Gaussian cho φ của các shadow IN và OUT. Online: `log N(φ_target; IN) − log N(φ_target; OUT)`. Offline: z-score so với OUT, `(φ_target − μ_out)/σ_out`. Mỗi bản ghi chỉ có K/2 điểm nên dùng **phương sai chung** (global) cho IN và OUT.
- **RMIA**: `p(x) = exp(-MSE(x)/τ)`, `τ` = median MSE của shadow OUT (hằng số toàn cục). `Ratio(x) = p_target(x)/p̄(x)`, score = tỉ lệ mẫu z trong tập tham chiếu Z có `Ratio(x)/Ratio(z) ≥ γ`. Offline: `p̄(x) = ½((1+a)·p_out(x) + (1−a))`. Online: `p̄(x) = ½(p_in(x) + p_out(x))`. `a`, `γ` lấy từ meta của split; Z là `ref_ids` của file mask, bỏ các id thuộc tập val.
- Bảng độ nhạy τ ∈ {0.5, 1, 2} × median chỉ để kiểm tra; kết quả ghi ra file luôn dùng τ × 1.0 (chọn trước, không chọn theo kết quả).

In [ ]:
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc as sk_auc

EPS = 1e-8
S_mse = np.stack([torch.load(f'{CKPT_DIR}/shadow_{LEVEL}_s{SEED}_k{j}_mse.pt')['mse'].numpy() for j in range(K)]).astype(np.float64)   # (K, N)
print('shadow mse', S_mse.shape)

# target per-record MSE placed on the pool (val rows stay NaN: the target was never evaluated there)
def rec_mse(preds, ds):
    return ((preds.numpy().astype(np.float64) - ds.timeseries[..., LOOKBACK:]) ** 2).mean(axis=(1, 2))
T_mse = np.full(N, np.nan)
idx_m, idx_n = np.arange(n_m), np.arange(n_m + n_v, N)
T_mse[idx_m], T_mse[idx_n] = rec_mse(tgt['pm'], member_ds), rec_mse(tgt['pn'], nonmem_ds)

idx_t = np.r_[idx_m, idx_n]                 # evaluated samples: members then non-members
Min = masks_pool.astype(float); Mout = 1 - Min
PHI_S, PHI_T = -np.log(S_mse + EPS), -np.log(T_mse + EPS)

def group_stats(W, X):
    c = W.sum(0)
    mu = (W * X).sum(0) / c
    return mu, (W * (X - mu) ** 2).sum(0) / (c - 1)

def lira(online):
    mu_in, v_in = group_stats(Min, PHI_S)
    mu_out, v_out = group_stats(Mout, PHI_S)
    s_in, s_out = np.sqrt(v_in[idx_t].mean()), np.sqrt(v_out[idx_t].mean())   # global std (few shadows per sample)
    x = PHI_T[idx_t]
    if online:
        return norm.logpdf(x, mu_in[idx_t], s_in) - norm.logpdf(x, mu_out[idx_t], s_out)
    return (x - mu_out[idx_t]) / s_out

TAU0 = float(np.median(S_mse[Mout.astype(bool)]))     # median MSE of OUT shadow predictions: attacker-side only
def rmia(online, tau_mult=1.0):
    tau = TAU0 * tau_mult
    P_S, P_T = np.exp(-S_mse / tau), np.exp(-T_mse / tau)
    p_in = (Min * P_S).sum(0) / Min.sum(0)
    p_out = (Mout * P_S).sum(0) / Mout.sum(0)
    p_bar = 0.5 * (p_in + p_out) if online else 0.5 * ((1 + RMIA_A) * p_out + (1 - RMIA_A))
    ratio = P_T / p_bar
    z = ratio[np.intersect1d(ref_pool, idx_t)]                                 # Z: ref ids that the target has predictions for
    return np.searchsorted(np.sort(z), ratio[idx_t] / RMIA_GAMMA, side='right') / len(z)

scores = {'lira': lira(True), 'lira_offline': lira(False), 'rmia': rmia(False), 'rmia_online': rmia(True)}
print({k: v.shape for k, v in scores.items()}, '|Z| =', len(np.intersect1d(ref_pool, idx_t)), 'tau0 =', round(TAU0, 5))

In [ ]:
def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0

def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))

def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))],
                                sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]

mse_m, mse_n = float(T_mse[idx_m].mean()), float(T_mse[idx_n].mean())
METHOD = {'lira': 'LiRA online on -log(MSE), global variance', 'lira_offline': 'LiRA offline z-score on -log(MSE), global variance',
          'rmia': f'RMIA offline a={RMIA_A} gamma={RMIA_GAMMA} p=exp(-MSE/tau)', 'rmia_online': f'RMIA online gamma={RMIA_GAMMA} p=exp(-MSE/tau)'}

os.makedirs(RESULTS_DIR, exist_ok=True)
rows = []
for a, s in scores.items():
    sm, sn = s[:n_m], s[n_m:]
    t01, t1, au = mia_metrics(sm, sn)
    c01, c1 = bootstrap_ci(sm, sn)
    out = {
        'model': 'foundation', 'task': 'fcst', 'attack': a,
        'owner_tv': 4, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
        'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
        'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
        'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
        'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
        'utility': {'train_mse': mse_m, 'test_mse': mse_n, 'gap': mse_n - mse_m},
        'n_shadow': K,
        'extra': {'mode': MODE, 'method': METHOD[a], 'epochs': EPOCHS, 'eval_epoch': TARGET_EPOCH, 'max_lr': MAX_LR,
                  'batch': BATCH, 'backbone': MODEL_NAME, 'lookback': LOOKBACK, 'horizon': HORIZON, 'tau': TAU0, 'split_sha256': meta['sha256'],
                  'shadow_masks_sha256': meta['shadow']['sha256']},
    }
    fn = f'{RESULTS_DIR}/{a}_{MODE}.json'
    json.dump(out, open(fn, 'w'))
    rows.append({'attack': a, 'tpr@0.1': t01, 'tpr@0.1 CI': c01, 'tpr@1': t1, 'tpr@1 CI': c1, 'auc': au})
    print('saved', fn)
pd.DataFrame(rows)

# RMIA sensitivity to tau (diagnostic only; saved files above always use tau multiplier 1.0)
sens = []
for tm in TAU_MULTS:
    for online in (False, True):
        s = rmia(online, tm)
        t01, t1, au = mia_metrics(s[:n_m], s[n_m:])
        sens.append({'rmia': 'online' if online else 'offline', 'tau_mult': tm, 'tpr@0.1': t01, 'tpr@1': t1, 'auc': au})
print(pd.DataFrame(sens).round(4).to_string(index=False))

## 5. So sánh tất cả attack
Gộp Loss và Temporal (từ `moment_fcst_colab` và `moment_temporal_colab`) với LiRA / RMIA. ROC log–log.

In [ ]:
import matplotlib.pyplot as plt
names = ['loss', 'temporal', 'lira', 'lira_offline', 'rmia', 'rmia_online']
fig, ax = plt.subplots(figsize=(6, 6))
tab = []
for a in names:
    p = f'{RESULTS_DIR}/{a}_{MODE}.json'
    if not os.path.exists(p): continue
    r = json.load(open(p))
    sm, sn = np.array(r['scores_member']), np.array(r['scores_nonmember'])
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    ax.plot(fpr, tpr, label=f"{a} (AUC {r['auc']:.3f})")
    tab.append({'attack': a, 'auc': r['auc'], 'tpr@0.1%': r['tpr_at_0.1fpr'], 'tpr@1%': r['tpr_at_1fpr'], 'n_shadow': r['n_shadow']})
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.set_title(f'MOMENT PTB-XL fcst h={HORIZON}, {LEVEL}-level')
plt.show()
pd.DataFrame(tab)